# Evaluation of the effect of RAG and Chain-of-Thought on the performance of large language models

* Load the packages and the MMLU datasets

In [1]:

#This notebook will evaluate different answer generation functions on the mmlu dataset, summarizing the results and outputting them to CSV files.

import pandas as pd
import general_utils
#Import other answer generation functions as needed
from evaluate_model_fast import batch_eval
from rag import rag_pipeline, load_vector_store
from summarize_accuracy import summarize_accuracy
import os

# load Gemini client
client = general_utils.load_gemini_client()

# MMLU datasets to evaluate
dataset_names = ['anatomy', 'formal_logic', 'moral_scenarios']

# Define prompts we want to try
prompt_db = {"standard": "Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. ",
             "cot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_max_thinking": "Take your time and think carefully before answering. Use the maximum amount of thinking, don’t rush - I want your best answer. It’s better to think too much than too little. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Provide a summary of your reasoning in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_emotional_manipulation": "Your ability to correctly answer this question is extremely important to my career. I will lose my job if you do not answer correctly. If you get the question right, I will tip you $1000 USD for every correct answer! Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_medical_career": "I'm a medical professional. Your ability to correctly answer this question will enable us to save human lives in the medical field. Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_persona": "You are an expert in \{domain\} with 50 years of working experience. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_few_shot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field. Below are example questions and correct answers: \n\n",
             "rag": "Please only use the provided content to answer the question above. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field.",
             "rag_optional": "You have the option to use the provided additional content to answer the question above. Please feel free to disregard the content if it is not helpful or not relevant. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field."
             }

# Range of k values to try for RAG
k_range = range(1,9,1)

# Range of few shot examples to include in prompt
few_shot_range = range(1,6,1)

# round number
round_num = 5


/Users/skysheng/miniforge3/envs/rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/skysheng/miniforge3/envs/rag/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


In [ ]:
round_num=5
all_files = os.listdir(f"../results/round{round_num}")
all_csv = [file for file in all_files if file.endswith(".csv")]
master_df = pd.DataFrame()

for file in all_csv:
    df = pd.read_csv(os.path.join(f"../results/round{round_num}", file))
    master_df = pd.concat([master_df, df])

master_df.to_csv(f'../results/round{round_num}/master_results_df.csv', index=False)


81000
